In [84]:
import os, sys

sys.path.append(os.path.abspath(os.path.join(os.path.dirname("utils"), "..")))
from module.utils import *
from module.prompt import *
from module.custom_model import *
from module.tools import *
from module.db_agent import *
from module.data_analysis_agent import *
from module.conversaction_agent import *

from typing_extensions import TypedDict

from typing import Annotated, List, Literal, Tuple
from langgraph.graph.message import add_messages
from langgraph.graph import StateGraph, START, END
from langchain_core.messages import (
    AIMessage,
    HumanMessage,
    SystemMessage,
    ToolMessage,
    RemoveMessage,
)
from langgraph.prebuilt import create_react_agent
from langchain_core.tools import tool
from langchain_core.runnables import RunnableLambda, RunnableWithFallbacks
from langgraph_supervisor import create_supervisor


class State(TypedDict):
    question: Annotated["str", "human request to llm"]
    summary: Annotated[str, "previous messages summarization "]
    messages: Annotated[list, add_messages]  # 화면 출력용
    history: Annotated[list, add_messages]  # 지난 행동 기록용
    answer: Annotated["str", "llm generate answer"]


def get_prompt_summary():
    template = """ 
        #System :
        As a summary-only node in LangGraph, you are responsible for summarizing key information by receiving records from previous conversations.  
        Be sure to follow the following rules:

        # Rules:
        1. If the user's requirements are **Analyzing, Comparison, Evaluation, and Modification requests for answers generated by previous LLM:
        - Never summarize the answers generated by the previous LLM and keep the original ****.
        - Combine content by briefly summarizing other surrounding contexts (e.g., discussion process, question background, etc.)

        2. For common conversations (information queries, descriptions, requests, etc.) that do not meet the above conditions:
        - Only important contents are summarized in **sentence form.**.
        - Remove unnecessary detailed descriptions, duplicate sentences, and text repeated in context.
        - It is described focusing on facts, and opinions or expressions of emotions are excluded.

        3. Finally, the summary is written in Korean.

        # Enter :
        - User Request : {question}
        - LLM's previous answer : {answer}
        - All previous conversations : {messages}
        - Summary of the conversation : {summary}
    """
    return ChatPromptTemplate.from_template(template)


def summary_node(state: State):
    """
    이전 모든 대화 내용중 핵심을 포함하여 요약하기위한 노드
    messages 가 6개 이상인 경우 summarization 수행
    """
    question = state.get("question", "")
    messages = state.get("messages", "")
    answer = state.get("answer", "")
    summary = state.get("summary", "")
    chain = get_prompt_summary() | get_gemini()
    print(f"summary :  \n\n")
    response = chain.invoke(
        {
            "question": question,
            "messages": messages,
            "answer": answer,
            "summary": summary,
        }
    )
    # 오래된 메시지 삭제
    delete_messages = [RemoveMessage(id=m.id) for m in state["messages"][:-2]]
    delete_history = [RemoveMessage(id=m.id) for m in state["messages"]]
    print(f"summary : {response} \n\n")
    # 요약 정보 반환
    return {
        "summary": response.content,
        "messages": delete_messages,
        "history": delete_history,
    }


def supervisor_node(state: State):
    """
    두개의 에이전트를 자율적으로 선택하여 사용하는 관리자에이전트 구조
    """
    summary = state.get("summary", "")
    question = state["question"]
    content = f"Previous Conversations: {summary} \n Human Request : {question}"
    db = get_db_agent()
    conversation = get_conversation_agent()
    data_analysis = get_data_analysis_agent()
    supervisor = create_supervisor(
        model=get_gpt(),
        agents=[db, conversation, data_analysis],
        prompt=(
            "You are a supervisor managing three agents:\n"
            "- db_agent: assign database select,insert,delete,update tasks\n"
            "- conversation_agent: assign normal conversaction and no use agent\n"
            "- data_analysis_agent: Agents that generate and run Python code for data analysis\n"
            "# Important : you must final answer is Korean\n"
        ),
        add_handoff_back_messages=True,
        output_mode="full_history",
    ).compile()
    inputs = {"messages": [{"role": "user", "content": content}]}
    config = get_runnable_config(recursion_limit=15, thread_id=get_random_uuid())
    print(f" supervisor : \n")
    # for chunk in supervisor.stream(inputs, config,stream_mode="values",subgraphs=True):
    #     print(f" supervisor : {chunk} \n\n")
    #     return chunk
    for step, metadata in supervisor.stream(inputs, config, stream_mode="messages"):
        # print(f" supervisor : {step} \n\n")
        # print(f" supervisor : {metadata} \n\n")
        # print(f" -------------------------------------------------- \n\n")
        return {'step':step,'metadata':metadata}

    # result = supervisor.invoke(inputs, {"recursion_limit": 15})
    # messages = result["messages"]
    # return {"messages": messages, "history": messages, "answer": messages[-1].content}


def get_graph():
    state_graph = StateGraph(State)
    state_graph.add_node("summary_node", summary_node)
    state_graph.add_node("supervisor_node", supervisor_node)

    state_graph.add_edge(START, "summary_node")
    state_graph.add_edge("summary_node", "supervisor_node")
    state_graph.add_edge("supervisor_node", END)

    cp = get_check_pointer()
    return state_graph.compile(checkpointer=cp)


def get_config():
    return get_runnable_config(recursion_limit=10, thread_id=get_random_uuid())

In [95]:
graph = get_graph()
config = get_config()

# user_input = "artist 테이블 10개만 조회해줘"
user_input = "Music 에서 가장 인기많은 곡 3개만 가져다줘"
inputs = {"question": user_input}
for step, metadata in graph.stream(input=inputs, config=config, stream_mode="messages",subgraphs=True,
):  

    # print(metadata)
    # print(metadata[0].content, end="")
    if step == ():
        print(1)
    print(f"main : {step} \n")
    print(f"main_metadata: {metadata} \n\n")
    if isinstance(metadata, tuple):
        msg = metadata[0]
        if isinstance(msg, ToolMessage):
            print("ToolMessage입니다.")
        else:
            print("ToolMessage가 아닙니다.")
    # if metadata["langgraph_node"] == "supervisor_node" and (text := step.text()):
    #         print(text, end="")

# for chunk in graph.stream(inputs, config,stream_mode="values"):
#     print(chunk)o

summary :  


1
main : () 

main_metadata: (AIMessageChunk(content='사용', additional_kwargs={}, response_metadata={'safety_ratings': []}, id='run--9172ea22-80bc-4513-8665-f618d52409db', usage_metadata={'input_tokens': 263, 'output_tokens': 1, 'total_tokens': 264, 'input_token_details': {'cache_read': 0}}), {'thread_id': 'b3d158e5-48d2-45fb-98b0-30df20b6e9b1', 'langgraph_step': 1, 'langgraph_node': 'summary_node', 'langgraph_triggers': ('branch:to:summary_node',), 'langgraph_path': ('__pregel_pull', 'summary_node'), 'langgraph_checkpoint_ns': 'summary_node:825da932-db0a-1eb5-ff06-c35512e986d7', 'checkpoint_ns': 'summary_node:825da932-db0a-1eb5-ff06-c35512e986d7', 'ls_provider': 'google_genai', 'ls_model_name': 'gemini-2.5-flash-lite', 'ls_model_type': 'chat', 'ls_temperature': 0.0, 'ls_max_tokens': 1096}) 


ToolMessage가 아닙니다.
1
main : () 

main_metadata: (AIMessageChunk(content='자는 음악에서 가장 인기 있는 곡 3개를 요청했습니다.', additional_kwargs={}, response_metadata={'finish_reason': 'STOP', 'model_nam